In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


batch_processed_path = (
    "s3://stock-market-pipeline-mule-2026/"
    "processed/batch/stock_prices/"
)

streaming_processed_path = (
    "s3://stock-market-pipeline-mule-2026/"
    "processed/streaming/stock_quotes/"
)

daily_metrics_path = (
    "s3://stock-market-pipeline-mule-2026/"
    "curated/daily_stock_metrics/"
)

latest_quotes_path = (
    "s3://stock-market-pipeline-mule-2026/"
    "curated/latest_stock_quotes/"
)

batch_df = spark.read.parquet(batch_processed_path)
streaming_df = spark.read.parquet(streaming_processed_path)

symbol_window = Window.partitionBy("ticker").orderBy("trade_date")
window_20_days = symbol_window.rowsBetween(-19, 0)
window_50_days = symbol_window.rowsBetween(-49, 0)

daily_metrics_df = (
    batch_df
    .withColumn(
        "previous_close_price",
        F.lag("close_price").over(symbol_window),
    )
    .withColumn(
        "daily_return_percent",
        F.when(
            F.col("previous_close_price").isNotNull(),
            (
                (F.col("close_price") - F.col("previous_close_price"))
                / F.col("previous_close_price")
            ) * 100,
        ),
    )
    .withColumn(
        "daily_range_percent",
        (
            (F.col("high_price") - F.col("low_price"))
            / F.col("low_price")
        ) * 100,
    )
    .withColumn(
        "moving_average_20",
        F.avg("close_price").over(window_20_days),
    )
    .withColumn(
        "moving_average_50",
        F.avg("close_price").over(window_50_days),
    )
    .withColumn(
        "volatility_20",
        F.stddev_samp("daily_return_percent").over(window_20_days),
    )
    .withColumn("year", F.year("trade_date"))
)

daily_metrics_df.write.mode("overwrite").partitionBy("year").parquet(
    daily_metrics_path
)

latest_quote_window = Window.partitionBy("symbol").orderBy(
    F.col("market_event_time").desc()
)

latest_quotes_df = (
    streaming_df
    .withColumn(
        "rank",
        F.row_number().over(latest_quote_window),
    )
    .filter(F.col("rank") == 1)
    .drop("rank")
)

latest_quotes_df.write.mode("overwrite").parquet(latest_quotes_path)

display(
    daily_metrics_df
    .filter(F.col("ticker") == "AAPL")
    .select(
        "ticker",
        "trade_date",
        "close_price",
        "daily_return_percent",
        "moving_average_20",
        "moving_average_50",
        "volatility_20",
    )
    .orderBy(F.col("trade_date").desc())
    .limit(10)
)

ticker,trade_date,close_price,daily_return_percent,moving_average_20,moving_average_50,volatility_20
AAPL,2026-02-20,264.58,1.5350372246526978,264.896,265.90900000000005,2.045102450826684
AAPL,2026-02-19,260.58,-1.426139587667879,264.073,266.1700000000001,2.0257190972371566
AAPL,2026-02-18,264.35,0.17811126269517483,263.41499999999996,266.5288000000001,1.985565633660779
AAPL,2026-02-17,263.88,3.166783954961293,262.521,266.8506000000001,2.162320492454492
AAPL,2026-02-13,255.78,-2.273335116341274,262.09150000000005,267.2506000000001,2.0594339297203765
AAPL,2026-02-12,261.73,-4.998185117967326,262.201,267.8534000000001,1.9975462652534446
AAPL,2026-02-11,275.5,0.6650102309266271,262.10049999999995,268.2756000000001,1.612935027072105
AAPL,2026-02-10,273.68,-0.3422911659748007,261.366,268.33740000000006,1.6105004372219343
AAPL,2026-02-09,274.62,-1.1660548477650647,260.6824999999999,268.4096,1.6040883935062902
AAPL,2026-02-06,277.86,0.8017413386541037,259.90799999999996,268.4514,1.5674022674735335
